# 02 — Full-scale HF data → train → smoke → Hub → Space

Kaggle **GPU (T4)** path with **Hugging Face auth via `HF_TOKEN` secret**:

1. Load `HF_TOKEN` from Kaggle Secrets (auth for dataset stream + later publish)
2. Stream **capped full-scale samples** from public HF datasets (not multi-GB dumps)
3. Build grounded SFT set → full-epoch QLoRA
4. Smoke adapter → push model → deploy Gradio Space

| Flag | Default | Meaning |
|------|---------|---------|
| `DOWNLOAD_HF` | `True` | Stream from Hub with token auth |
| `MAX_PER_SOURCE` | transcripts 400 / fiqa 200 / alpaca 150 | Portfolio-scale caps |
| `RUN_TRAIN` | `True` | Full epoch on the built set |
| `PUBLISH_HF` / `PUBLISH_SPACE` | `True` | After smoke |

**Secret name must be exactly `HF_TOKEN`.** Write scope for publish; read is enough for public datasets.

This is **not** the entire S&P corpus on disk — it is a **streaming cap** designed to produce a multi-step SFT run (thousands of grounded pairs after chunk/generate).

## 0. Knobs

In [ ]:
RUN_TRAIN = True
MAX_STEPS = None                 # None = full epoch over the built train split

# --- Full-scale public HF ingest (token required for reliable auth) ---
DOWNLOAD_HF = True
# Per-source streaming caps (portfolio full run; not infinite dump)
MAX_PER_SOURCE = {
    "earnings_transcripts": 400,
    "fiqa": 200,
    "finance_alpaca": 150,
}
USE_LLM_JUDGE = False
CONFIG_PATH = "configs/default.yaml"

RUN_SMOKE = True
SMOKE_PROMPT = (
    "Summarize prepared remarks vs Q&A on a US large-cap quarterly earnings call."
)

RUN_SIDE_BY_SIDE = True
SIDE_BY_SIDE_LIMIT = 4

PUBLISH_HF = True
HF_REPO_ID = "nuwanda94/llama32-3b-ecra-sft"
HF_PRIVATE = False

PUBLISH_SPACE = True
SPACE_REPO_ID = "nuwanda94/earnings-call-research-assistant"
SPACE_PRIVATE = False
SPACE_DIR = "spaces/ecra-demo"

LAUNCH_GRADIO = False
GRADIO_SHARE = True
GRADIO_SIDE_BY_SIDE = True

ADAPTER_DIR = "outputs/adapters/llama32-3b-ecra-sft"
SMOKE_OK = False

print("DOWNLOAD_HF", DOWNLOAD_HF, "MAX_PER_SOURCE", MAX_PER_SOURCE)
print("RUN_TRAIN", RUN_TRAIN, "MAX_STEPS", MAX_STEPS)
print("PUBLISH_HF", PUBLISH_HF, HF_REPO_ID)
print("PUBLISH_SPACE", PUBLISH_SPACE, SPACE_REPO_ID)

## 1. Repo path + installs

In [ ]:
from pathlib import Path
import os
import sys

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

if IN_KAGGLE:
    work = Path("/kaggle/working")
    repo = work / "earnings-call-research-assistant"
    # Always refresh so ingest/train fixes are present
    %cd /kaggle/working
    !rm -rf earnings-call-research-assistant
    !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
    REPO = (work / "earnings-call-research-assistant").resolve()
else:
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
assert (SRC / "earnings_call_research_assistant" / "inference.py").exists(), SRC
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO:", REPO)

import earnings_call_research_assistant as ecra
print("package:", ecra.__file__)

In [ ]:
if IN_KAGGLE:
    %pip install -q pyyaml huggingface_hub datasets
    if RUN_TRAIN or RUN_SMOKE or RUN_SIDE_BY_SIDE or LAUNCH_GRADIO:
        %pip install -q unsloth transformers accelerate bitsandbytes trl peft
    if LAUNCH_GRADIO:
        %pip install -q gradio

## 2. Load HF_TOKEN from Kaggle Secrets (before any Hub call)

Add-ons → Secrets → name **`HF_TOKEN`**. Used for dataset streaming **and** adapter/Space publish.

In [ ]:
from earnings_call_research_assistant.data import wire_hf_token, resolve_hf_token

def _load_hf_token() -> bool:
    if resolve_hf_token():
        return wire_hf_token()
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
            if tok and tok.strip():
                return wire_hf_token(tok.strip())
        except Exception as e:
            print("Kaggle secrets note:", type(e).__name__, str(e)[:160])
    return False

has_token = _load_hf_token()
print("HF token active:", has_token, "(value never printed)")
if DOWNLOAD_HF and not has_token:
    print("WARNING: DOWNLOAD_HF=True but no token — public streams may 429 more often.")
if (PUBLISH_HF or PUBLISH_SPACE) and not has_token:
    print("WARNING: publish enabled but no HF_TOKEN.")
assert has_token or not DOWNLOAD_HF, (
    "Set Kaggle secret HF_TOKEN for full-scale Hub ingest."
)

## 3. Phase 1 — full-scale public ingest (HF streaming + caps)

Streams each source up to `MAX_PER_SOURCE` with your token. Then chunk → grounded pairs → filter → versioned splits.

In [ ]:
from earnings_call_research_assistant.data import (
    DATASET_VERSION, DEFAULT_MAX_PER_SOURCE, ChunkConfig, FilterConfig,
    GenerateConfig, SelectConfig, chunk_records, filter_pairs, generate_pairs,
    ingest_catalog, list_sources, select_and_split, write_chunks_jsonl,
    write_filter_report, write_jsonl, write_pairs_jsonl, write_splits,
)

for s in list_sources():
    print(f"  - {s.source_id}: {s.display_name} ({s.hf_id})")

caps = {**DEFAULT_MAX_PER_SOURCE, **MAX_PER_SOURCE}
print("Using per-source caps:", caps)
print("DOWNLOAD_HF:", DOWNLOAD_HF, "| token:", bool(resolve_hf_token()))

records = ingest_catalog(
    max_per_source=caps,
    download=DOWNLOAD_HF,
    pause_between_sources_s=2.0,  # reduce 429 risk between datasets
)
write_jsonl(records, Path("data/raw/public_sample.jsonl"))
print("records:", len(records))
from collections import Counter
print("by source:", dict(Counter(r.source_id for r in records)))

chunks = chunk_records(records, config=ChunkConfig(window_sentences=4, stride_sentences=2))
write_chunks_jsonl(chunks, Path("data/processed/chunks.jsonl"))
print("chunks:", len(chunks), "props:", sum(len(c.propositions) for c in chunks))

pairs = generate_pairs(
    chunks,
    config=GenerateConfig(max_qa_per_chunk=2, include_summary=True, use_llm=False),
)
write_pairs_jsonl(pairs, Path("data/processed/grounded_pairs.jsonl"))
print("pairs:", len(pairs))

kept, report = filter_pairs(
    pairs,
    config=FilterConfig(
        min_output_chars=40,
        near_dup_jaccard=0.88,
        use_llm_judge=USE_LLM_JUDGE,
        min_judge_score=0.6,
    ),
)
write_pairs_jsonl(kept, Path("data/processed/filtered_pairs.jsonl"))
write_filter_report(report, Path("data/processed/filter_report.json"))
print("kept:", report.n_kept, "dropped:", report.dropped_by_stage)

OUT_DIR = Path("data/processed") / DATASET_VERSION
sel_cfg = SelectConfig(
    target_min=100,
    target_max=6000,
    max_per_source=2500,
    diversity_jaccard_cap=0.72,
    seed=94,
    dataset_version=DATASET_VERSION,
)
splits, sel_report = select_and_split(kept, config=sel_cfg)
paths = write_splits(splits, OUT_DIR, report=sel_report, config=sel_cfg)
print(
    f"selected={sel_report.n_selected} train={sel_report.n_train} "
    f"val={sel_report.n_val} test={sel_report.n_test}"
)
print("train path:", paths["train"])
assert sel_report.n_train >= 20, (
    f"Train split too small ({sel_report.n_train}). Check ingest caps / HF errors."
)

## 4. SFT dry-run plan

In [ ]:
from earnings_call_research_assistant.training.sft import run_sft
import json

plan = run_sft(
    config_path=CONFIG_PATH,
    dataset_dir=OUT_DIR,
    dry_run=True,
    max_steps=MAX_STEPS,
    require_train=False,
)
print(f"model={plan.model_name} train={plan.n_train} val={plan.n_val} adapter={plan.adapter_dir}")
print("Expect many optimizer steps when n_train is hundreds+.")

## 5. Full QLoRA train (one epoch over the built train split)

Uses `num_train_epochs` from YAML when `MAX_STEPS is None`. With hundreds of train rows you should see **many steps**, not `[1/1]`.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

if not RUN_TRAIN:
    print("Skipped train.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable T4 GPU.")
    if plan.n_train < 20:
        raise RuntimeError(
            f"Train set too small ({plan.n_train}). Re-run Phase 1 with DOWNLOAD_HF=True + HF_TOKEN."
        )
    train_plan = run_sft(
        config_path=CONFIG_PATH,
        dataset_dir=OUT_DIR,
        dry_run=False,
        max_steps=MAX_STEPS,
        require_train=True,
    )
    print("Train finished:", train_plan.adapter_dir)
    print("notes:", train_plan.notes[-5:])

adapter_path = Path(ADAPTER_DIR)
print("adapter exists:", adapter_path.exists())
if adapter_path.exists():
    print("files:", sorted(p.name for p in adapter_path.iterdir())[:15])

## 6. Smoke-generate with adapter (gates Hub)

In [ ]:
import gc
import yaml
from earnings_call_research_assistant.inference import InferenceConfig, InferenceHarness

def _release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

SMOKE_OK = False
smoke_reply = ""

if not RUN_SMOKE:
    print("RUN_SMOKE=False.")
elif not Path(ADAPTER_DIR).exists():
    raise FileNotFoundError(ADAPTER_DIR)
elif not torch.cuda.is_available():
    raise RuntimeError("Smoke needs CUDA.")
else:
    with Path(CONFIG_PATH).open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    harness = None
    try:
        try:
            harness = InferenceHarness.from_pretrained(cfg, model_name=ADAPTER_DIR)
        except Exception as e:
            print("Direct load failed, base+load_adapter:", e)
            harness = InferenceHarness.from_pretrained(cfg)
            harness.model.load_adapter(ADAPTER_DIR)
        smoke_reply = (harness.generate(SMOKE_PROMPT) or "").strip()
        print("SMOKE REPLY:")
        print(smoke_reply[:800] if smoke_reply else "(empty)")
        if len(smoke_reply) >= 20:
            SMOKE_OK = True
        else:
            raise RuntimeError("Smoke reply too short.")
    finally:
        if harness is not None:
            del harness
        _release()
print("SMOKE_OK =", SMOKE_OK)

## 7. Optional side-by-side

In [ ]:
from earnings_call_research_assistant.eval.panel import load_panel, DEFAULT_PANEL

def _generate_batch(harness, items):
    rows = []
    for item in items:
        text = item.user_text()
        reply = harness.generate(text)
        rows.append({"id": item.id, "ticker": item.ticker, "theme": item.theme,
                     "user_text": text, "reply": reply})
        print(f"  [{item.id}] -> {len(reply)} chars")
    return rows

compare_path = Path("evals/reports/side_by_side_panel.jsonl")
compare_path.parent.mkdir(parents=True, exist_ok=True)

if not RUN_SIDE_BY_SIDE or not SMOKE_OK:
    print("Skipped side-by-side.")
else:
    panel = load_panel(DEFAULT_PANEL)[: max(1, int(SIDE_BY_SIDE_LIMIT))]
    with Path(CONFIG_PATH).open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    base_h = InferenceHarness.from_pretrained(cfg)
    base_rows = _generate_batch(base_h, panel)
    del base_h
    _release()
    try:
        tuned_h = InferenceHarness.from_pretrained(cfg, model_name=ADAPTER_DIR)
    except Exception as e:
        tuned_h = InferenceHarness.from_pretrained(cfg)
        tuned_h.model.load_adapter(ADAPTER_DIR)
    tuned_rows = _generate_batch(tuned_h, panel)
    del tuned_h
    _release()
    by_id = {r["id"]: r for r in tuned_rows}
    with compare_path.open("w", encoding="utf-8") as f:
        for br in base_rows:
            tr = by_id.get(br["id"], {})
            row = {"id": br["id"], "ticker": br["ticker"], "theme": br["theme"],
                   "user_text": br["user_text"], "base_reply": br["reply"],
                   "adapter_reply": tr.get("reply", "")}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("Wrote", compare_path)

## 8. Push adapter to HF Hub

In [ ]:
from earnings_call_research_assistant.publish import publish_adapter

if not PUBLISH_HF:
    print("Skipped adapter upload.")
elif not SMOKE_OK:
    raise RuntimeError("SMOKE_OK is False.")
else:
    live = publish_adapter(
        adapter_dir=ADAPTER_DIR, repo_id=HF_REPO_ID, private=HF_PRIVATE,
        commit_message="feat: upload ECRA QLoRA adapter (full-scale Kaggle run)",
        dry_run=False,
    )
    print("Adapter Hub:", live.hub_url)

## 9. Deploy Gradio Space

In [ ]:
from earnings_call_research_assistant.space_publish import publish_space

if not PUBLISH_SPACE:
    print("Skipped Space.")
elif not SMOKE_OK:
    raise RuntimeError("SMOKE_OK is False.")
else:
    space_live = publish_space(
        space_dir=SPACE_DIR, repo_id=SPACE_REPO_ID, private=SPACE_PRIVATE,
        commit_message="feat: deploy ECRA Space after full-scale train",
        dry_run=False,
    )
    print("Space URL:", space_live.hub_url)
    print("Settings → Hardware → T4; ADAPTER_REPO=", HF_REPO_ID)

## Done

Success signals:
- Phase 1 `records` in the hundreds+
- `train=` hundreds+
- Trainer progress **not** stuck at `[1/1]`
- Adapter + Space URLs printed